imports

In [31]:
from datetime import datetime, timedelta, timezone
from pydantic import BaseModel
from dataclasses import dataclass
from typing import Optional
from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api._errors import TranscriptsDisabled, NoTranscriptFound
import feedparser


In [32]:

class Transcript(BaseModel):
    text: str

In [33]:

class ChannelVideo(BaseModel):
    title: str
    url: str
    video_id: str
    published_at: datetime
    description: str
    transcript: Optional[str]= None

In [34]:
from typing import List


class YouTubeScraper:
    def __init__(self):
        
        self.transcript_api = YouTubeTranscriptApi()

    def _get_rss_url(self, channel_id: str) -> str:
        return f"https://www.youtube.com/feeds/videos.xml?channel_id={channel_id}"

    def _extract_video_id(self, video_url: str) -> str:
        if "youtube.com/watch?v=" in video_url:
            return video_url.split("v=")[1].split("&")[0]
        if "youtube.com/shorts/" in video_url:
            return video_url.split("shorts/")[1].split("?")[0]
        if "youtu.be/" in video_url:
            return video_url.split("youtu.be/")[1].split("?")[0]
        return video_url

    def get_transcript(self, video_id: str) -> Optional[Transcript]:
        try:
            transcript = self.transcript_api.fetch(video_id)
            text = " ".join([snippet.text for snippet in transcript.snippets])
            return Transcript(text=text)
        except (TranscriptsDisabled, NoTranscriptFound):
            return None
        except Exception:
            return None

    def get_latest_videos(self, channel_id: str, hours: int = 24) -> list[ChannelVideo]:
        feed = feedparser.parse(self._get_rss_url(channel_id))
        if not feed.entries:
            return []
        
        cutoff_time = datetime.now(timezone.utc) - timedelta(hours=hours)
        videos = []
        
        for entry in feed.entries:
            if "/shorts/" in entry.link:
                continue
            published_time = datetime(*entry.published_parsed[:6], tzinfo=timezone.utc)
            if published_time >= cutoff_time:
                video_id = self._extract_video_id(entry.link)
                videos.append(ChannelVideo(
                    title=entry.title,
                    url=entry.link,
                    video_id=video_id,
                    published_at=published_time,
                    description=entry.get("summary", "")
                ))
        
        return videos

    def scrape_channel(self, channel_id: str, hours: int = 150) -> list[ChannelVideo]:
        videos = self.get_latest_videos(channel_id, hours)
        result = []
        for video in videos:
            transcript = self.get_transcript(video.video_id)
            result.append(video.model_copy(update={"transcript": transcript.text if transcript else None}))
        return result
    
    


scraper = YouTubeScraper()

transcript: Transcript = scraper.get_transcript("jqd6_bbjhS8")
print(transcript.text)
channel_videos: List[ChannelVideo] = scraper.scrape_channel("UCn8ujwUInbJkBhffxqAPBVQ", hours=200)




All right. So, in this tutorial, I'm going to walk you through a common pattern that we see in AI development. And that is combining internal knowledge from, for example, a handbook with web search. And web search can either be a specific URL that we already know beforehand where we would just want to get the data from that page or a broader search where we let the agent go on and on in loops, collect the information, and then use that as context. So this is a request that we see more and more from our clients where they want to build internal agents that have a rack pipeline. They have access to internal knowledge but we add the web on top of that usually in fallback scenarios. So that's what I want to cover in this video. Now high level if we go to the image over here what that looks like is we'll build an agent that can take a user query that will then decide and the agent will then decide to say look I can look into the internal policies into the handbook that I have or I can eithe

In [1]:
import sys
for p in sys.path:
    print(p)



/home/rym/Documents/Study/IA/LLM_PROJECT/venvnews/lib/python312.zip
/home/rym/Documents/Study/IA/LLM_PROJECT/venvnews/lib/python3.12
/home/rym/Documents/Study/IA/LLM_PROJECT/venvnews/lib/python3.12/lib-dynload

/home/rym/Documents/Study/IA/LLM_PROJECT/venvnews/lib/python3.12/site-packages
